# Decision tree hyperparameters
max_depth on the Social Network Ads data, then every main hyperparameter on the two-moons data. The interactive version is `app.py`.

In [ ]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from app import fit   # loads the data, splits it and trains a tree (same code as the app)

## max_depth on the Social Network Ads data

In [ ]:
# Train one tree per depth; None means 'grow until every leaf is pure'
for depth in [None, 1, 2, 3, 4, 5]:
    tree, X_train, X_test, y_train, y_test, axes, classes = fit("ads", max_depth=depth)
    print(f"max_depth={depth}: {tree.get_n_leaves()} leaves, train {tree.score(X_train, y_train):.3f}, test {tree.score(X_test, y_test):.3f}")

In [ ]:
# Choose max_depth honestly: 5-fold cross-validation on the training set only
grid = GridSearchCV(DecisionTreeClassifier(random_state=42),
                    {"max_depth": [1, 2, 3, 4, 5, 6, 8, 10, None]}, cv=5)
grid.fit(X_train, y_train)
print(grid.best_params_, round(grid.best_score_, 3), "test:", round(grid.score(X_test, y_test), 3))

## Every hyperparameter on the two-moons data (375 training rows)

In [ ]:
settings = [{}, {"criterion": "entropy"}, {"splitter": "random"}, {"max_depth": 1}, {"max_depth": 2}, {"max_depth": 5},
            {"min_samples_split": 100}, {"min_samples_split": 301}, {"min_samples_leaf": 20}, {"min_samples_leaf": 100},
            {"max_features": 1}, {"max_leaf_nodes": 5}, {"min_impurity_decrease": 0.01}, {"min_impurity_decrease": 0.1}]
for params in settings:
    tree, X_train, X_test, y_train, y_test, axes, classes = fit("moons", **params)
    print(f"{str(params):32s} depth {tree.get_depth():2d}, {tree.get_n_leaves():3d} leaves, "
          f"train {tree.score(X_train, y_train):.3f}, test {tree.score(X_test, y_test):.3f}")

In [ ]:
# min_samples_split = 301: the root (375 rows) splits once, its children are too small to split again
tree, *_ = fit("moons", min_samples_split=301)
print(export_text(tree, feature_names=["x1", "x2"], show_weights=True))

In [ ]:
# min_impurity_decrease: the weighted decrease of the root split, as scikit-learn computes it
tree, *_ = fit("moons")
t = tree.tree_
N = t.n_node_samples[0]
L, R = t.children_left[0], t.children_right[0]
dec = t.n_node_samples[0] / N * (t.impurity[0]
       - t.n_node_samples[L] / t.n_node_samples[0] * t.impurity[L]
       - t.n_node_samples[R] / t.n_node_samples[0] * t.impurity[R])
print("root Gini", round(t.impurity[0], 3), "children", round(t.impurity[L], 3), round(t.impurity[R], 3),
      "sizes", t.n_node_samples[L], t.n_node_samples[R], "decrease", round(dec, 3))